### Transformer Training Preprocessing Steps
1. load all matrices in from prepared_data/
2. convert annotations to HAC label standard to be selected
3. Split into Train-Validation-Test -> using split mentioned in paper
4. Save all into .npys in in specified directory

In [1]:
import numpy as np
file_path = 'capture24/prepared_data_1024_uniform//'
# Let's take a look at the prepared data
X_from_npy = np.load(f'{file_path}/X.npy')
print(X_from_npy[10])

# Let's take a look at the labels
Y_from_npy = np.load(f'{file_path}/Y_anno.npy', allow_pickle=True)
print(Y_from_npy[10])

Y_W2020 = np.load(f'{file_path}/Y_Walmsley2020.npy')
print(Y_W2020[10])

Y_Willetts2018 = np.load(f'{file_path}/Y_WillettsSpecific2018.npy')
print(Y_Willetts2018[10])

# Let's take a look at the time stamps
T_from_npy = np.load(f'{file_path}/T.npy')
print(T_from_npy[:10])

# Let's take a look at the patient ids
P_from_npy = np.load(f'{file_path}/P.npy')
# Process data to remove P and convert to int
P_from_npy = [int(i.split('P')[1]) for i in P_from_npy]

[[ 0.8251444  -0.2780502  -0.48983255]
 [ 0.8251444  -0.2780502  -0.48983255]
 [ 0.84065473 -0.2780502  -0.48983255]
 ...
 [ 0.8251444  -0.2780502  -0.48983255]
 [ 0.8251444  -0.2780502  -0.47412065]
 [ 0.8251444  -0.2780502  -0.48983255]]
7030 sleeping;MET 0.95
sleep
sleep
['2016-04-10T22:54:00.000000000' '2016-04-10T22:54:10.240000000'
 '2016-04-10T22:54:20.480000000' '2016-04-10T22:54:30.720000000'
 '2016-04-10T22:54:40.960000000' '2016-04-10T22:54:51.200000000'
 '2016-04-10T22:55:01.440000000' '2016-04-10T22:55:11.680000000'
 '2016-04-10T22:55:21.920000000' '2016-04-10T22:55:32.160000000']


In [2]:
# Create Training - Test Split
# Capture 24 specified split as the following:
# Participants 001 - 100 -> Training - for final (001-080: Train, 081-100: Validation)
# Participants (P)101 - 150 -> Test

# Easiest way to split is perhaps utilize a pandas dataframe for filtering
import pandas as pd

df = pd.DataFrame(
    {
        "Accelerometer_data": [x for x in X_from_npy],
        "Raw_Labels": Y_from_npy,
        "Walmsley2020_Labels": Y_W2020,
        "Willetts2018_Labels": Y_Willetts2018,
        "Time_Stamps": T_from_npy,
        "Patient_ID": P_from_npy
    }
)


# Split into training and test - save under capture24/final_data/
train_df = df[(df['Patient_ID'] <= 79)].reset_index(drop=True)
valid_df = df[(df['Patient_ID'] > 79) & (df['Patient_ID'] <= 100)].reset_index(drop=True)
test_df = df[(df['Patient_ID'] > 100)].reset_index(drop=True)


In [3]:
import os
import json

# Create Index Mapping for the Labels
unique_labels = sorted(set(train_df['Willetts2018_Labels'].unique()))
label_to_index = {label: index for index, label in enumerate(unique_labels)}
index_to_label = {index: label for label, index in label_to_index.items()}


# Encode the Labels - let's use the Willetts2018_Labels - in numpy format
Y_train = np.array(train_df['Willetts2018_Labels'])
Y_train_id = np.array([label_to_index[label] for label in Y_train])

# Encode the validation labels in a similar manner
Y_valid = np.array(valid_df['Willetts2018_Labels'])
Y_valid_id = np.array([label_to_index[label] for label in Y_valid])

# Encode the test labels in a similar manner
Y_test = np.array(test_df['Willetts2018_Labels'])
Y_test_id = np.array([label_to_index[label] for label in Y_test])

dir_path = 'capture24/final_data_1024_uniform/'
os.makedirs(dir_path, exist_ok=True)
with open(f'{dir_path}/label_to_index.json', 'w') as f:
    json.dump({
        "label_to_index": label_to_index,
        "index_to_label": index_to_label
    }, f)


# Reshape the input data to be [B, T, C]
X_train = train_df["Accelerometer_data"].apply(
    lambda x: np.asarray(x, dtype=np.float32)
)

X_valid = valid_df["Accelerometer_data"].apply(
    lambda x: np.asarray(x, dtype=np.float32)
)

X_test = test_df["Accelerometer_data"].apply(
    lambda x: np.asarray(x, dtype=np.float32)
)


np.save(f'{dir_path}/X_train.npy', np.stack(X_train.values))
np.save(f'{dir_path}/X_valid.npy', np.stack(X_valid.values))
np.save(f'{dir_path}/X_test.npy', np.stack(X_test.values))

# Save the labels
np.save(f'{dir_path}/Y_train.npy', Y_train_id)
np.save(f'{dir_path}/Y_valid.npy', Y_valid_id)
np.save(f'{dir_path}/Y_test.npy', Y_test_id)

In [6]:
# Create a weights matrix of size (1, 10) - balanced weighting 
# get the count of each unique value 
import gzip
import numpy as np

with gzip.open('capture24/final_data_1024_mode_v/Y_train.npy.gz', 'rb') as f:
    y_train = np.load(f)

with gzip.open('capture24/final_data_1024_mode_v/Y_valid.npy.gz', 'rb') as f:
    y_valid = np.load(f)

with gzip.open('capture24/final_data_1024_mode_v/Y_test.npy.gz', 'rb') as f:
    y_test = np.load(f)

# combine all of the labels
y_all = np.concatenate([y_train, y_valid, y_test])
total = y_all.shape[0]

# get the count of each unique value
unique, counts = np.unique(y_all, return_counts=True)
print(unique, counts)

# create a weights matrix of size (1, 10) - balanced weighting 
weights = total / (counts * 10)
print(weights)

# save the weights
np.save('capture24/final_data_1024_mode_v/weights.npy', weights)






[0 1 2 3 4 5 6 7 8 9] [  8748  62897   9795  37698 321985 332023   5144  29197  33930  58637]
[10.28868313  1.43099671  9.18891271  2.3875378   0.2795329   0.27108182
 17.49716174  3.08269343  2.65267905  1.53495916]


In [4]:
# Quick Check on the difference between filtered and non-filtered
import gzip
import numpy as np
import pandas as pd
# get the length of two files
with gzip.open('capture24/final_data_1024/Y_test.npy.gz', 'rb') as f:
    y_train = np.load(f)

with gzip.open('capture24/final_data_1024_mode/Y_test.npy.gz') as f1:
    y_train_mode = np.load(f1)


print(f"Size: {y_train.shape}, Size w/filtering: {y_train_mode.shape}")
print(f"Difference {y_train.shape[0] - y_train_mode.shape[0]}")

# confirm how many unique values there are in both
print(np.unique(y_train))
print(np.unique(y_train_mode))

Y_anno = np.load('capture24/prepared_data_1024/Y_anno.npy', allow_pickle=True)
Y_anno_mode = np.load('capture24/prepared_data_1024_mode_filtered/Y_anno.npy', allow_pickle=True)

# Check for NA in raw annotations
print("NA count in original:", pd.isna(Y_anno).sum())
print("NA count in filtered:", pd.isna(Y_anno_mode).sum())


Size: (301323,), Size w/filtering: (296771,)
Difference 4552
[0 1 2 3 4 5 6 7 8 9]
[0 1 2 3 4 5 6 7 8 9]
NA count in original: 13054
NA count in filtered: 0
